In [1]:
import mne
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Qt5Agg')  # 使用Qt5后端支持交互

In [2]:
def print_edf_info(raw, file_name):
    """打印EDF文件的基本信息和通道信息"""
    try:
        print(f"\n=== Processing {file_name} ===")
        print("=== EDF File Information ===")
        print(f"File Duration: {raw.times[-1]:.2f} seconds")
        print(f"Start DateTime: {raw.info['meas_date']}")
        print(f"Number of Channels: {len(raw.info['ch_names'])}")
        print(f"Sampling Frequency: {raw.info['sfreq']:.2f} Hz")
        print(f"High-pass Filter: {raw.info['highpass']:.2f} Hz")
        print(f"Low-pass Filter: {raw.info['lowpass']:.2f} Hz")

        print("\n=== Channel Information ===")
        for i, ch_name in enumerate(raw.info['ch_names']):
            ch_type = mne.channel_type(raw.info, i)
            print(f"Channel {i}: Name: {ch_name}, Type: {ch_type}")

        if raw.info.get('subject_info'):
            print("\n=== Subject Information ===")
            for key, value in raw.info['subject_info'].items():
                if value is not None:
                    print(f"{key}: {value}")

    except Exception as e:
        print(f"Error printing EDF info for {file_name}: {str(e)}")

In [3]:
def bandpass_filter(raw: mne.io.Raw, l_freq: float = 1.0, h_freq: float = 40.0) -> mne.io.Raw:
    """对 EEG 信号应用带通滤波（参考 CAUEEG_preprocessing）。"""
    try:
        sfreq = raw.info["sfreq"]
        if l_freq is not None and l_freq < 0:
            raise ValueError("l_freq 必须 >= 0")
        if h_freq is not None and h_freq >= sfreq / 2:
            h_freq = sfreq / 2 - 0.1
            print(f"警告：h_freq 太高，已自动调整为 {h_freq:.2f} Hz")
        raw_filtered = raw.copy().filter(
            l_freq=l_freq,
            h_freq=h_freq,
            picks="eeg",
            method="fir",
            phase="zero",
            verbose=False,
        )
        print(f"应用带通滤波：{l_freq} Hz - {h_freq} Hz")
        return raw_filtered
    except Exception as e:
        print(f"错误：带通滤波失败: {e}")
        return raw


def downsample(raw: mne.io.Raw, target_sfreq: float) -> mne.io.Raw:
    """对 EEG 信号降采样（参考 CAUEEG_preprocessing：MNE resample）。"""
    try:
        original_sfreq = float(raw.info["sfreq"])
        if target_sfreq >= original_sfreq:
            print(f"警告：目标采样率 {target_sfreq} Hz 不低于原始采样率 {original_sfreq} Hz，跳过降采样")
            return raw
        raw_downsampled = raw.copy().resample(sfreq=target_sfreq, npad="auto", verbose=False)
        print(f"降采样：{original_sfreq} Hz → {target_sfreq} Hz")
        return raw_downsampled
    except Exception as e:
        print(f"错误：降采样失败: {e}")
        return raw

In [4]:
def plot_scroll_signal(raw):
    """显示10秒信号窗口，支持时间进度条"""
    try:
        print("\nGenerating scrollable signal plot (10s window, filtered)...")
        raw.plot(
            n_channels=len(raw.info['ch_names']),  # 显示所有通道
            duration=10.0,  # 每个窗口显示10秒
            scalings='auto',  # 自动调整Y轴缩放
            title='Filtered EEG Signals (0.5-45Hz, 10s Window)',
            show=True,
            block=True,  # 保持窗口打开直到手动关闭
            show_scrollbars=True,  # 显示时间进度条
            time_format='float'  # 时间轴显示为秒
        )
    except Exception as e:
        print(f"Error plotting signal: {str(e)}")

In [5]:
def window_qc(
    windows,
    sfreq,
    n_channels,
    bad_channel_ratio_thresh=0.2,  # >20% 通道坏 → 丢窗
    abs_amp_thresh=200.0,  # µV，保护性绝对阈值
    verbose=False,
):
    """
    对窗口进行质量检测，识别异常窗口。

    参数:
    - windows (np.ndarray): 窗口数据，3D数组，形状为 (n_windows, window_length, channels) 或 (n_windows, channels, window_length)
    - sfreq (float): 采样率 (Hz)
    - n_channels (int): 通道数，用于判断和修正维度顺序
    - bad_channel_ratio_thresh (float): 坏通道比例阈值，超过此比例的窗口将被标记为异常。默认: 0.2
    - abs_amp_thresh (float): 绝对幅值阈值 (µV)。默认: 200.0
    - verbose (bool): 是否打印详细信息。默认: False

    返回:
    - qc_pass (np.ndarray): 布尔数组，形状为 (n_windows,)，True 表示通过 QC
    """
    if windows.ndim != 3:
        raise ValueError(f"windows 必须是 3D 数组，当前形状: {windows.shape}")

    original_shape = windows.shape

    if windows.shape[-1] == n_channels:
        windows = np.transpose(windows, (0, 2, 1))
        if verbose:
            print(f"  形状修正: {original_shape} -> {windows.shape} (交换后两个维度)")

    n_w, n_ch, n_t = windows.shape

    ptp = np.ptp(windows, axis=-1)
    ptp_med = np.median(ptp, axis=0)
    ptp_mad = np.median(np.abs(ptp - ptp_med), axis=0) + 1e-6
    ptp_bad = ptp > (ptp_med + 6 * ptp_mad)

    absmax = np.max(np.abs(windows), axis=-1)
    abs_bad = absmax > abs_amp_thresh

    bad_channel_mask = ptp_bad | abs_bad
    bad_ratio = bad_channel_mask.sum(axis=1) / n_ch
    if verbose:
        print(f"  坏道数量: {bad_channel_mask.sum(axis=1)}")

    qc_pass = bad_ratio <= bad_channel_ratio_thresh

    return qc_pass


def segment_random_crop_pair_array(
    data,
    window_length,
    step_length,
    sfreq,
    n_channels,
    enable_qc=True,
    rng=None,
    random_seed=42,
    verbose=False,
):
    """
    Random Cropping 双路成对增强：生成两条窗口序列（原始/增强），并保证窗口一一对应。

    返回:
        (windows_ori, windows_aug, meta)
        - windows_ori/windows_aug: shape=(N, window_length, n_channels)
        - meta: 包含 start_ori/start_aug/gap/n_windows/stride/L_work 等信息

    说明:
        - 两路使用相同 stride (=step_length) 与相同 n_windows
        - QC 时按索引成对过滤：仅当同一下标两窗均通过才保留
    """
    if rng is None:
        rng = np.random.default_rng(random_seed)

    if data.ndim == 1:
        data = data.reshape(-1, 1)
    elif data.ndim != 2:
        raise ValueError(f"数据维度应为1D或2D，但得到 {data.ndim}D")

    T, C = data.shape
    W = int(window_length)
    stride = int(step_length)
    stride = max(1, stride)

    empty_meta = {
        "start_ori": None,
        "start_aug": None,
        "gap": None,
        "n_windows": 0,
        "reserved_tail": None,
        "stride": stride,
        "L_work": None,
    }

    if T < W or C != int(n_channels):
        return np.empty((0, W, C)), np.empty((0, W, C)), empty_meta

    last_start = ((T - W) // stride) * stride
    tail = T - (last_start + W)

    if tail == 0:
        reserved_tail = W
    elif tail < (W / 2.0):
        reserved_tail = 2 * W
    else:
        reserved_tail = W

    L_work = T - reserved_tail
    if L_work < W:
        return np.empty((0, W, C)), np.empty((0, W, C)), {**empty_meta, "reserved_tail": reserved_tail, "L_work": L_work}

    n_windows = (L_work - W) // stride + 1
    max_start = L_work - W - (n_windows - 1) * stride
    if max_start < 0:
        return np.empty((0, W, C)), np.empty((0, W, C)), {**empty_meta, "reserved_tail": reserved_tail, "L_work": L_work}

    d_min = int(np.ceil(W / 10.0))
    d_max = int(np.floor(W / 2.0))
    if d_min > d_max:
        raise ValueError(
            f"window_length={W} 过小：无法满足起点间隔 ∈ [W/10, W/2]（得到 d_min={d_min}, d_max={d_max}）"
        )

    if max_start == 0:
        start_ori = 0
        start_aug = 0
        gap = 0
    else:
        d_lo, d_hi = d_min, d_max
        if max_start < d_min:
            d_lo = 1
            d_hi = min(d_max, max_start)
        if d_lo > d_hi:
            start_ori = 0
            start_aug = 0
            gap = 0
        else:
            start_ori = None
            start_aug = None
            gap = None
            for _ in range(500):
                g = int(rng.integers(d_lo, d_hi + 1))
                upward = rng.random() < 0.5
                if upward:
                    lo, hi = 0, max_start - g
                    if lo > hi:
                        continue
                    start_ori = int(rng.integers(lo, hi + 1))
                    start_aug = start_ori + g
                else:
                    lo, hi = g, max_start
                    if lo > hi:
                        continue
                    start_ori = int(rng.integers(lo, hi + 1))
                    start_aug = start_ori - g
                gap = g
                break

            if start_ori is None:
                raise RuntimeError(
                    "无法在约束下采样到合法的 (start_ori, start_aug)；请检查 window_length、step_length 与数据长度。"
                )

    windows_ori = np.empty((n_windows, W, C), dtype=data.dtype)
    windows_aug = np.empty((n_windows, W, C), dtype=data.dtype)
    for k in range(n_windows):
        i_ori = start_ori + k * stride
        i_aug = start_aug + k * stride
        windows_ori[k] = data[i_ori : i_ori + W, :]
        windows_aug[k] = data[i_aug : i_aug + W, :]

    meta = {
        "start_ori": int(start_ori),
        "start_aug": int(start_aug),
        "gap": int(gap),
        "n_windows": int(n_windows),
        "reserved_tail": int(reserved_tail),
        "L_work": int(L_work),
        "stride": int(stride),
    }

    if enable_qc and n_windows > 0:
        qc_ori = window_qc(windows_ori, sfreq, n_channels, verbose=verbose)
        qc_aug = window_qc(windows_aug, sfreq, n_channels, verbose=verbose)
        qc_pair = qc_ori & qc_aug
        if verbose:
            print(
                f"  [random_crop_pair] QC: 总窗口 {n_windows}, 两路均通过 {qc_pair.sum()}, 丢弃 {n_windows - qc_pair.sum()}"
            )
        windows_ori = windows_ori[qc_pair]
        windows_aug = windows_aug[qc_pair]
        meta["qc_pass_ori"] = qc_ori
        meta["qc_pass_aug"] = qc_aug
        meta["qc_pass_pair"] = qc_pair

    return windows_ori, windows_aug, meta


def sliding_window(data: np.ndarray, window_size: int, step_size: int, channels: list,
                   file_name: str) -> np.ndarray:
    """对EEG数据应用滑窗分割，从丢弃部分长度的一半开始，仅保留完整窗口"""
    try:
        if data.shape[1] < window_size:
            print(f"警告：{file_name} 数据长度 {data.shape[1]} 小于窗口长度 {window_size}，返回空数组")
            return np.array([]).reshape(0, data.shape[0], window_size)

        # 计算完整窗口数量
        num_windows = (data.shape[1] - window_size) // step_size + 1
        # 计算丢弃部分长度
        discarded_length = data.shape[1] - num_windows * window_size
        # 计算起始点（丢弃部分长度的一半）
        start_idx = discarded_length // 2

        # 检查从起始点开始是否能生成至少一个完整窗口
        if start_idx + window_size > data.shape[1]:
            print(
                f"警告：{file_name} 从起始点 {start_idx} 开始无法生成完整窗口（数据长度 {data.shape[1]}，丢弃部分 {discarded_length}）")
            return np.array([]).reshape(0, data.shape[0], window_size)

        # 计算从起始点开始的完整窗口数量
        windows = []
        for i in range(start_idx, start_idx + num_windows * step_size, step_size):
            window = data[:, i:i + window_size]
            if window.shape[1] == window_size:  # 确保窗口长度为256
                windows.append(window)

        if not windows:
            print(f"警告：{file_name} 从起始点 {start_idx} 开始没有生成任何完整窗口（数据长度 {data.shape[1]}，丢弃部分 {discarded_length}）")
            return np.array([]).reshape(0, data.shape[0], window_size)

        # 打印每个窗口的统计信息
        print(f"\n{file_name}: 数据长度 {data.shape[1]}，丢弃部分 {discarded_length}，起始点 {start_idx}，生成 {len(windows)} 个窗口")
        for idx, window in enumerate(windows):
            power = np.mean(window ** 2, axis=1)  # 计算均值功率
            print(f"Window {idx + 1}:")
            for ch, p in zip(channels, power):
                print(f"  Channel {ch}: Mean power = {p:.2e}")

        return np.stack(windows, axis=0).transpose(0, 2, 1) # 形状: (样本数, 通道数, 窗口长度)
    except Exception as e:
        print(f"Error in sliding window processing for {file_name}: {str(e)}")
        return np.array([]).reshape(0, data.shape[0], window_size)

In [6]:
def load_labels(states_file, file_names):
    """从states.xlsx加载标签，映射NORMAL=0，MCI=1"""
    try:
        df = pd.read_excel(states_file, sheet_name='Sheet1')
        labels = {}
        for file_name in file_names:
            # 假设文件名（不含扩展名）在Excel的'file number'列中匹配
            base_name = os.path.splitext(os.path.basename(file_name))[0]
            label_row = df[df['file number'] == int(base_name)]
            if not label_row.empty:
                label = label_row['status'].iloc[0]
                if label == 'NORMAL':
                    labels[file_name] = 0
                elif label == 'MCI':
                    labels[file_name] = 1
                else:
                    print(f"警告：{file_name} 的标签 {label} 无效，跳过")
                    labels[file_name] = None
            else:
                print(f"警告：{file_name} 在 {states_file} 中未找到标签，跳过")
                labels[file_name] = None
        return labels
    except Exception as e:
        print(f"Error loading labels from {states_file}: {str(e)}")
        return {file_name: None for file_name in file_names}

In [7]:
def process_edf_files(
    data_dir,
    states_file,
    output_dir,
    file_counter,
    all_labels,
    channels_ref,
    sfreq_ref,
    *,
    window_size=256,
    step_size=256,
    use_augmentation=False,
    augmentation_random_seed=42,
    enable_window_qc=True,
    enable_windowing=True,
    target_sampling_rate=128.0,
    bandpass_l_freq=1.0,
    bandpass_h_freq=40.0,
    original_sampling_rate=None,
):
    """处理指定目录下的所有EDF文件：带通滤波 → 降采样 → 可选划窗与窗口 QC，保存每个文件的结果。"""
    try:
        edf_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith(".edf")]
        if not edf_files:
            print(f"警告：{data_dir} 中未找到EDF文件")
            return file_counter, all_labels, channels_ref, sfreq_ref

        labels = load_labels(states_file, edf_files)

        for file_path in edf_files:
            print(f"\n=== Processing file: {file_path} ===")
            raw = mne.io.read_raw_edf(file_path, preload=True)

            original_sfreq = float(raw.info["sfreq"])
            if original_sampling_rate is not None and abs(original_sfreq - float(original_sampling_rate)) > 1.0:
                print(
                    f"提示：{os.path.basename(file_path)} 采样率 {original_sfreq:.2f} Hz 与 "
                    f"ORIGINAL_SAMPLING_RATE={original_sampling_rate} 不一致（仍继续处理）"
                )

            raw_bp = bandpass_filter(raw, l_freq=bandpass_l_freq, h_freq=bandpass_h_freq)
            raw_ds = downsample(raw_bp, target_sampling_rate)

            data = raw_ds.get_data(picks="eeg")
            sfreq = float(raw_ds.info["sfreq"])
            channels = [
                ch
                for ch, ch_type in zip(raw_ds.info["ch_names"], raw_ds.get_channel_types())
                if ch_type == "eeg"
            ]

            if channels_ref is None:
                channels_ref = channels
                sfreq_ref = sfreq
            elif channels != channels_ref or abs(sfreq - float(sfreq_ref)) > 0.01:
                print(f"警告：{file_path} 的通道或采样率与参考不一致，跳过")
                raw.close()
                raw_bp.close()
                raw_ds.close()
                continue

            n_ch = data.shape[0]
            data_tc = np.asarray(data).T

            if not enable_windowing:
                full = data_tc[np.newaxis, ...]  # (1, T, C)
                if full.shape[1] == 0:
                    full = np.empty((0, 0, n_ch), dtype=data.dtype)
                elif enable_window_qc:
                    qc_pass = window_qc(full, sfreq, n_ch, verbose=False)
                    kept = bool(qc_pass[0])
                    print(f"  质量控制(整段): {'通过' if kept else '丢弃'}")
                    if not kept:
                        full = np.empty((0, data_tc.shape[0], n_ch), dtype=data.dtype)
                if use_augmentation:
                    if full.shape[0] > 0:
                        windows = np.stack([full, full.copy()], axis=0)
                    else:
                        windows = np.empty((2, 0, data_tc.shape[0], n_ch), dtype=data.dtype)
                else:
                    windows = full
            elif use_augmentation:
                ori, aug, _meta = segment_random_crop_pair_array(
                    data_tc,
                    window_length=window_size,
                    step_length=step_size,
                    sfreq=sfreq,
                    n_channels=n_ch,
                    enable_qc=enable_window_qc,
                    random_seed=augmentation_random_seed,
                    verbose=False,
                )
                if len(ori) > 0:
                    windows = np.stack([ori, aug], axis=0)
                else:
                    windows = np.empty((2, 0, window_size, n_ch), dtype=data.dtype)
            else:
                windows = sliding_window(
                    data,
                    window_size=window_size,
                    step_size=step_size,
                    channels=channels,
                    file_name=file_path,
                )
                if enable_window_qc and windows.size > 0:
                    qc_pass = window_qc(windows, sfreq, n_ch, verbose=False)
                    windows = windows[qc_pass]

            if windows.size > 0:
                file_counter += 1
                output_file = os.path.join(output_dir, f"feature_{file_counter:03d}.npy")
                np.save(output_file, windows)
                print(f"保存滑窗数据到 {output_file}，形状: {windows.shape}")

                label = labels[file_path]
                if label is not None:
                    all_labels.append([label, file_counter])

            raw.close()
            raw_bp.close()
            raw_ds.close()

        return file_counter, all_labels, channels_ref, sfreq_ref

    except Exception as e:
        print(f"Error processing files in {data_dir}: {str(e)}")
        return file_counter, all_labels, channels_ref, sfreq_ref

In [8]:
def main():
    """主函数，处理 EEG_1 / EEG_2，保存滑窗特征与标签（参数与路径对齐 ADFTD_preprocessing）。"""
    try:
        # ---------- 数据根目录（含 EEG_1、EEG_2、states.xlsx）----------
        data_root = r"/root/autodl-tmp/MCI-normal Sub61"
        eeg1_dir = os.path.join(data_root, "EEG_1")
        eeg2_dir = os.path.join(data_root, "EEG_2")
        eeg1_states = os.path.join(eeg1_dir, "states.xlsx")
        eeg2_states = os.path.join(eeg2_dir, "States.xlsx")

        # ---------- 重采样参数（对齐 ADFTD）----------
        ORIGINAL_SAMPLING_RATE = 256  # 原始采样率 (Hz)，用于与 EDF 实际 sfreq 对照提示
        TARGET_SAMPLING_RATE = 128  # 目标采样率 (Hz)；降采样实现与 CAUEEG 一致（MNE resample）
        # ADFTD：'linear'/'cubic'/'nearest'；本脚本降采样实现与 CAUEEG 一致（MNE），与 ADFTD 的 scipy 插值不同
        RESAMPLING_KIND = "linear"

        # ---------- 带通（先带通再降采样；默认对齐 CAUEEG 示例 1–40 Hz，可按需改）----------
        BANDPASS_L_FREQ = 0.5
        BANDPASS_H_FREQ = 45.0

        # ---------- 窗口分割参数（对齐 ADFTD）----------
        WINDOW_TIME = 1.0
        OVERLAP_RATIO = 0.0
        WINDOW_SIZE = int(WINDOW_TIME * TARGET_SAMPLING_RATE)
        STEP_SIZE = max(1, int(round(WINDOW_SIZE * (1.0 - OVERLAP_RATIO))))
        # 是否进行划窗；False 时不切分，整段重采样序列作为一个样本
        ENABLE_WINDOWING = False
        # 是否进行窗口质量检测；False 时保留全部窗口（未划窗时则对整段检测）
        ENABLE_WINDOW_QC = False

        # ---------- 数据增强（Random Cropping 双路）；True 时保存 (2, N, WINDOW_SIZE, n_channels) ----------
        # 关闭划窗时 Random Crop 不生效，两路保存同一整段
        USE_AUGMENTATION = False
        AUGMENTATION_RANDOM_SEED = 42

        # ---------- 输出路径（默认划窗 + QC 时目录名与原来一致；关闭时追加后缀，避免覆盖已有结果）----------
        output_base_dir = r"/root/autodl-tmp/ADMamba/dataset"
        output_tags = []
        if not ENABLE_WINDOWING:
            output_tags.append("NoWin")
        if not ENABLE_WINDOW_QC:
            output_tags.append("NoQC")
        if USE_AUGMENTATION:
            output_tags.append("Aug")
        tag_str = ("_" + "_".join(output_tags)) if output_tags else ""
        output_dir_name = f"IUMSMCI_{WINDOW_TIME}s_{OVERLAP_RATIO}{tag_str}"
        OUTPUT_DIR = os.path.join(output_base_dir, output_dir_name)
        OUTPUT_LABEL_DIR = os.path.join(OUTPUT_DIR, "Label")
        OUTPUT_FEATURE_DIR = os.path.join(OUTPUT_DIR, "Feature")

        # LABEL_MAP = {"A": 2, "F": 1, "C": 0}  # ADFTD 用；IUMSMCI 标签由 load_labels 解析 states.xlsx

        os.makedirs(OUTPUT_FEATURE_DIR, exist_ok=True)
        os.makedirs(OUTPUT_LABEL_DIR, exist_ok=True)

        print("IUMSMCI 预处理参数")
        print(f"  数据根目录: {data_root}")
        print(
            f"  ORIGINAL_SAMPLING_RATE={ORIGINAL_SAMPLING_RATE}, TARGET_SAMPLING_RATE={TARGET_SAMPLING_RATE}, "
            f"RESAMPLING_KIND={RESAMPLING_KIND}（实现为 CAUEEG 同款 MNE resample）"
        )
        print(f"  带通: {BANDPASS_L_FREQ}-{BANDPASS_H_FREQ} Hz → 降采样至 {TARGET_SAMPLING_RATE} Hz")
        print(f"  WINDOW_TIME={WINDOW_TIME}s, OVERLAP_RATIO={OVERLAP_RATIO}, WINDOW_SIZE={WINDOW_SIZE}, STEP_SIZE={STEP_SIZE}")
        print(f"  ENABLE_WINDOWING={ENABLE_WINDOWING}, ENABLE_WINDOW_QC={ENABLE_WINDOW_QC}, USE_AUGMENTATION={USE_AUGMENTATION}")
        if USE_AUGMENTATION and not ENABLE_WINDOWING:
            print("  注意: 已关闭划窗，Random Crop 增强不生效，两路保存同一整段")
        print(f"  OUTPUT_DIR={OUTPUT_DIR}")

        file_counter = 0
        all_labels = []
        channels_ref = None
        sfreq_ref = None

        kw = dict(
            window_size=WINDOW_SIZE,
            step_size=STEP_SIZE,
            use_augmentation=USE_AUGMENTATION,
            augmentation_random_seed=AUGMENTATION_RANDOM_SEED,
            enable_window_qc=ENABLE_WINDOW_QC,
            enable_windowing=ENABLE_WINDOWING,
            target_sampling_rate=float(TARGET_SAMPLING_RATE),
            bandpass_l_freq=BANDPASS_L_FREQ,
            bandpass_h_freq=BANDPASS_H_FREQ,
            original_sampling_rate=ORIGINAL_SAMPLING_RATE,
        )

        if os.path.exists(eeg1_dir):
            print(f"\n=== Processing EEG_1 directory: {eeg1_dir} ===")
            file_counter, all_labels, channels_ref, sfreq_ref = process_edf_files(
                eeg1_dir,
                eeg1_states,
                OUTPUT_FEATURE_DIR,
                file_counter,
                all_labels,
                channels_ref,
                sfreq_ref,
                **kw,
            )
        else:
            print(f"警告：{eeg1_dir} 不存在")

        if os.path.exists(eeg2_dir):
            print(f"\n=== Processing EEG_2 directory: {eeg2_dir} ===")
            file_counter, all_labels, channels_ref, sfreq_ref = process_edf_files(
                eeg2_dir,
                eeg2_states,
                OUTPUT_FEATURE_DIR,
                file_counter,
                all_labels,
                channels_ref,
                sfreq_ref,
                **kw,
            )
        else:
            print(f"警告：{eeg2_dir} 不存在")

        if all_labels:
            labels_file = os.path.join(OUTPUT_LABEL_DIR, "label.npy")
            np.save(labels_file, np.array(all_labels))
            print(f"保存所有标签到 {labels_file}，形状: {np.array(all_labels).shape}")
        else:
            print("警告：未生成任何标签")

    except Exception as e:
        print(f"Error in main: {str(e)}")

In [9]:
if __name__ == "__main__":
    main()

IUMSMCI 预处理参数
  数据根目录: /root/autodl-tmp/MCI-normal Sub61
  ORIGINAL_SAMPLING_RATE=256, TARGET_SAMPLING_RATE=128, RESAMPLING_KIND=linear（实现为 CAUEEG 同款 MNE resample）
  带通: 0.5-45.0 Hz → 降采样至 128 Hz
  WINDOW_TIME=1.0s, OVERLAP_RATIO=0.0, WINDOW_SIZE=128, STEP_SIZE=128
  ENABLE_WINDOWING=False, ENABLE_WINDOW_QC=False, USE_AUGMENTATION=False
  OUTPUT_DIR=/root/autodl-tmp/ADMamba/dataset/IUMSMCI_1.0s_0.0_NoWin_NoQC

=== Processing EEG_1 directory: /root/autodl-tmp/MCI-normal Sub61/EEG_1 ===

=== Processing file: /root/autodl-tmp/MCI-normal Sub61/EEG_1/1.edf ===
Extracting EDF parameters from /root/autodl-tmp/MCI-normal Sub61/EEG_1/1.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 465663  =      0.000 ...  1818.996 secs...
应用带通滤波：0.5 Hz - 45.0 Hz
降采样：256.0 Hz → 128.0 Hz
保存滑窗数据到 /root/autodl-tmp/ADMamba/dataset/IUMSMCI_1.0s_0.0_NoWin_NoQC/Feature/feature_001.npy，形状: (1, 232832, 19)

=== Processing file: /root/autodl-tmp/MCI-normal Sub61/EEG_1/2.edf ===
Ext

In [10]:
path_feature = '/root/autodl-tmp/ADMamba/dataset/IUMSMCI_1.0s_0.0_Aug/Feature'
all_samples = 0
for file in os.listdir(path_feature):
    sub_path = os.path.join(path_feature, file)
    shape = np.load(sub_path).shape
    # 增强数据 layout: (2, N, T, C)，第 0 维为增强视角，仅统计第 1 维原始样本数 N
    if len(shape) == 4 and shape[0] == 2:
        n_samples = shape[1]
    else:
        n_samples = shape[0]
    print(file, shape, '-> 原始样本数:', n_samples)
    all_samples += n_samples
print('原始样本总数:', all_samples)

path_label = '/root/autodl-tmp/ADMamba/dataset/IUMSMCI_1.0s_0.0_Aug/Label'
label_path = os.path.join(path_label, 'label.npy')
labels = np.load(label_path)
print('label 条目数:', len(labels))
print(labels)

feature_001.npy (2, 1719, 128, 19) -> 原始样本数: 1719
feature_002.npy (2, 1713, 128, 19) -> 原始样本数: 1713
feature_003.npy (2, 1746, 128, 19) -> 原始样本数: 1746
feature_004.npy (2, 1701, 128, 19) -> 原始样本数: 1701
feature_005.npy (2, 1716, 128, 19) -> 原始样本数: 1716
feature_006.npy (2, 1709, 128, 19) -> 原始样本数: 1709
feature_007.npy (2, 1719, 128, 19) -> 原始样本数: 1719
feature_008.npy (2, 1824, 128, 19) -> 原始样本数: 1824
feature_009.npy (2, 1973, 128, 19) -> 原始样本数: 1973
feature_010.npy (2, 1686, 128, 19) -> 原始样本数: 1686
feature_011.npy (2, 1807, 128, 19) -> 原始样本数: 1807
feature_012.npy (2, 1961, 128, 19) -> 原始样本数: 1961
feature_013.npy (2, 1810, 128, 19) -> 原始样本数: 1810
feature_014.npy (2, 1715, 128, 19) -> 原始样本数: 1715
feature_015.npy (2, 1771, 128, 19) -> 原始样本数: 1771
feature_016.npy (2, 2033, 128, 19) -> 原始样本数: 2033
feature_017.npy (2, 1711, 128, 19) -> 原始样本数: 1711
feature_018.npy (2, 1802, 128, 19) -> 原始样本数: 1802
feature_019.npy (2, 1740, 128, 19) -> 原始样本数: 1740
feature_020.npy (2, 2010, 128, 19) -> 原始样本数: 2010
